# Streaming a HATS catalog with `LSDBStreamDataset`

`LSDBStreamDataset` feeds a HATS catalog into Hyrax's `train_stream` and `infer_stream`
verbs through [LSDB](https://docs.lsdb.io), one chunk of partitions at a time. Nothing is
ever fully materialized, so the catalog can be far larger than memory.

This notebook covers:

1. pointing the dataset at a catalog, including one you derived interactively;
2. why derived catalogs go through a registry instead of the config;
3. training on the stream, and how ragged LSDB chunks become uniform batches;
4. choosing between a finite (`"catalog"`) and an endless (`"infinite"`) stream.

In [1]:
import itertools
import tempfile

import lsdb
import torch.nn as nn
import torch.nn.functional as F

import hyrax
from hyrax.datasets import LSDBStreamDataset
from hyrax.models.model_registry import hyrax_model

## 1. Open a HATS catalog with LSDB

You can start with a standard HATS catalog, or derive one with LSDB using queries, cone search, or crossmatches.
Here we take a magnitude cut on Gaia DR3.

In [2]:
gaia = lsdb.open_catalog(
    "https://data.lsdb.io/hats/gaia_dr3",
    columns=["ra", "dec", "phot_g_mean_mag"],
)
bright = gaia.query("phot_g_mean_mag < 19")
gaia

,ra,dec,phot_g_mean_mag
npartitions=2016,,,
"Order: 2, Pixel: 0",double[pyarrow],double[pyarrow],float[pyarrow]
...,...,...,...
"Order: 3, Pixel: 766",...,...,...
"Order: 3, Pixel: 767",...,...,...


Similarly we can open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [3]:
def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

tess_filtered_nans = tess.map_partitions(drop_nans)

## 2. Register a derived catalog

`bright` only exists in memory — there is no path or URL that names it. It also cannot be
dropped into the config directly: when a verb starts, Hyrax writes the whole runtime
configuration to `runtime_config.toml`, and a live `lsdb.Catalog` object is not
serializable.

So instead you register the catalog under a name and use the handle it returns as the
`data_location`. The handle is an ordinary string, so the config stays serializable.

In [4]:
data_location = LSDBStreamDataset.register_catalog("gaia_bright", bright)
data_location

'lsdb://gaia_bright'

In [5]:
data_location = LSDBStreamDataset.register_catalog("tess_filtered_nans", tess_filtered_nans)
data_location

'lsdb://tess_filtered_nans'

## 3. A small model for catalog columns

The models that ship with Hyrax expect images, so here is a compact dense autoencoder over
the two catalog columns.

For more information about defining custom models, see the [Hyrax documentation](https://hyrax.readthedocs.io/en/latest/).

In [6]:
@hyrax_model
class CatalogAutoencoder(nn.Module):
    """A tiny dense autoencoder over a handful of catalog columns."""

    def __init__(self, config, data_sample=None):
        super().__init__()
        self.config = config
        n_features = data_sample.shape[1]
        self.encoder = nn.Sequential(nn.Linear(n_features, 8), nn.GELU(), nn.Linear(8, 2))
        self.decoder = nn.Sequential(nn.Linear(2, 8), nn.GELU(), nn.Linear(8, n_features))

    def forward(self, batch):
        return self.encoder(batch)

    def train_batch(self, batch):
        reconstructed = self.decoder(self(batch))
        loss = F.mse_loss(reconstructed, batch)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return {"loss": loss.item()}

    def infer_batch(self, batch):
        return self(batch)

    @staticmethod
    def prepare_inputs(data_dict) -> tuple:
        """Stack the requested catalog columns into one (batch, n_features) array."""
        import numpy as np

        data = data_dict["data"]
        # Rescale each column to roughly [-1, 1]. Without this the raw ranges
        # (ra 0-360, dec -90-90) make the loss diverge immediately.
        ra = np.asarray(data["ra_obj"], dtype="float32") / 180.0 - 1.0
        dec = np.asarray(data["dec_obj"], dtype="float32") / 90.0
        return np.stack([ra, dec], axis=1)

## 4. Train on the stream

After update the configuration settings to make use of the model defined above
and specifying that the TESS catalog should be used in the data_request, we can begin training.

Note that `stream_type = "infinite"` keeps batches coming indefinitely, which is what you usually
want for training. We take five batches with `islice` and stop.

In [7]:
h = hyrax.Hyrax()
h.config["model"]["name"] = "CatalogAutoencoder"
h.config["general"]["results_dir"] = tempfile.mkdtemp()
h.config["data_loader"]["batch_size"] = 4

ds_config = h.config["data_set"]["LSDBStreamDataset"]
ds_config["stream_type"] = "infinite"
ds_config["partitions_per_chunk"] = 1

h.config["data_request"] = {
    "train_stream": {
        "data": {
            "dataset_class": "LSDBStreamDataset",
            "data_location": data_location,
            "primary_id_field": "ticid",
            "fields": ["ra_obj", "dec_obj", "lightcurve_sap_flux"],
        }
    }
}

with h.train_stream() as session:
    for batch, metrics in itertools.islice(session, 5):
        print(f"Batch length: {len(batch['object_id'])}    loss={metrics['loss']:.4f}")
        print(f"    {batch}")

[2026-09-18 11:46:17,236 hyrax.datasets.lsdb_stream_dataset:INFO] No active dask.distributed Client was found, so lsdb will compute each chunk synchronously and every fetch will block the consumer. Create a client before starting the run - Client(processes=False) keeps results in-process and avoids serializing every chunk back from a worker.
[2026-09-18 11:46:35,929 hyrax.datasets.data_provider:WARNING] Input data contains NaN values. This may mean your model output is all NaNs. Consider setting config['data_set']['nan_mode'] = 'quantile' or 'zero' or writing a to_tensor() function for your model. Search hyrax readthedocs for 'to_tensor' to get started.
[2026-09-18 11:46:35,930 hyrax.models.model_registry:INFO] Setting model's self.optimizer from config: torch.optim.SGD with arguments: {'lr': 0.01, 'momentum': 0.9}.
[2026-09-18 11:46:35,930 hyrax.models.model_registry:INFO] Setting model's self.criterion from config: torch.nn.CrossEntropyLoss with default arguments.
[2026-09-18 11:46:3

Batch length: 4    loss=0.8096
    {'object_id': array(['27530971', '27643046', '27457135', '27530971'], dtype='<U8'), 'data': {'ra_obj': array([295.59012, 295.74   , 295.28848, 295.59012], dtype=float32), 'dec_obj': array([49.1986  , 49.011517, 49.379784, 49.1986  ], dtype=float32), 'lightcurve_sap_flux': array([[         nan,          nan,          nan, ...,   0.        ,
          0.        ,   0.        ],
       [         nan,          nan,          nan, ...,   0.        ,
          0.        ,   0.        ],
       [         nan,          nan,          nan, ...,   0.        ,
          0.        ,   0.        ],
       [         nan,          nan,          nan, ..., 719.76843262,
        719.0289917 , 717.94378662]], shape=(4, 19232)), 'lightcurve_sap_flux_mask': array([[ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ...,  True,  True,  Tru

## 5. Ragged chunks, uniform batches

LSDB hands back one DataFrame per *chunk of partitions*, and partitions vary wildly in size
— for example, a Gaia partition holds hundreds of thousands of rows, while a crossmatched
catalog might give you nine. 
`LSDBStreamDataset` buffers rows across chunks and splits large ones, so every
batch is exactly `batch_size`.
Only the final batch of a finite stream is ever short.

Compare what LSDB yields against what the dataset yields:

In [8]:
raw_chunk = next(iter(lsdb.streams.CatalogStream(bright, partitions_per_chunk=1, seed=0)))
print(f"One raw LSDB chunk:  {len(raw_chunk):,} rows")

# The dataset can also be driven directly, without a verb.
dataset = LSDBStreamDataset(h.config, data_location=data_location)
print("Hyrax batch sizes:  ", [len(b) for b in itertools.islice(dataset, 5)])
dataset.stop()

[2026-09-18 11:46:41,277 hyrax.datasets.lsdb_stream_dataset:INFO] No active dask.distributed Client was found, so lsdb will compute each chunk synchronously and every fetch will block the consumer. Create a client before starting the run - Client(processes=False) keeps results in-process and avoids serializing every chunk back from a worker.


One raw LSDB chunk:  202,682 rows
Hyrax batch sizes:   [4, 4, 4, 4, 4]


## 6. `"catalog"` versus `"infinite"`

* **`stream_type = "catalog"`** wraps `lsdb.streams.CatalogStream`: a single finite pass
  that visits every object exactly once and then ends on its own. This is what you want for
  **inference**. Hyrax will disallow shuffling for this stream type.
* **`stream_type = "infinite"`** wraps `lsdb.streams.InfiniteStream`: partitions are
  resampled forever, so batches keep arriving until you stop. This is what you want for
  **training**, since it is not bounded by one pass over the catalog. Shuffling
  is optional with this stream type.

A finite pass is easiest to see on a small local catalog. Note the last batch is short —
that is the one place a partial batch is allowed.

In [9]:
import pandas as pd
import numpy as np

frame = pd.DataFrame(
    {
        "object_id": np.asarray([f"{i:03d}" for i in range(30)]),
        "ra_obj": np.linspace(0.0, 350.0, 30),
        "dec_obj": np.linspace(-80.0, 80.0, 30),
        "phot_g_mean_mag": np.linspace(14.0, 18.0, 30),
    }
)
small = lsdb.from_dataframe(frame, ra_column="ra_obj", dec_column="dec_obj")
small_location = LSDBStreamDataset.register_catalog("small", small)

h2 = hyrax.Hyrax()
h2.config["model"]["name"] = "CatalogAutoencoder"
h2.config["general"]["results_dir"] = tempfile.mkdtemp()
h2.config["data_loader"]["batch_size"] = 8
h2.config["data_set"]["LSDBStreamDataset"]["stream_type"] = "catalog"
h2.config["data_set"]["LSDBStreamDataset"]["shuffle"] = False
h2.config["data_request"] = {
    "train_stream": {
        "data": {
            "dataset_class": "LSDBStreamDataset",
            "data_location": small_location,
            "primary_id_field": "object_id",
            "fields": ["ra_obj", "dec_obj", "phot_g_mean_mag"],
        }
    }
}

seen = []
with h2.train_stream() as session:
    for batch, _metrics in session:
        seen.extend(batch["object_id"])
        print(f"batch of {len(batch['object_id'])} objects")

print(f"\nsaw {len(seen)} objects, {len(set(seen))} of them unique - the stream ended by itself")

[2026-09-18 11:47:09,207 hyrax.datasets.lsdb_stream_dataset:INFO] No active dask.distributed Client was found, so lsdb will compute each chunk synchronously and every fetch will block the consumer. Create a client before starting the run - Client(processes=False) keeps results in-process and avoids serializing every chunk back from a worker.
[2026-09-18 11:47:09,211 hyrax.models.model_registry:INFO] Setting model's self.optimizer from config: torch.optim.SGD with arguments: {'lr': 0.01, 'momentum': 0.9}.
[2026-09-18 11:47:09,211 hyrax.models.model_registry:INFO] Setting model's self.criterion from config: torch.nn.CrossEntropyLoss with default arguments.
[2026-09-18 11:47:09,212 hyrax.models.model_registry:INFO] Setting model's self.scheduler from config: torch.optim.lr_scheduler.ExponentialLR
with arguments: {'gamma': 1}.
2026-09-18 11:47:09,212 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<hyrax.datasets.stre': 
	{'batch_size': None, 'collate_fn':

batch of 8 objects
batch of 8 objects
batch of 8 objects
batch of 6 objects

saw 30 objects, 30 of them unique - the stream ended by itself


## 7. Limitations worth knowing

**Default support for nested columns.** A catalogs often carry nested
columns - containing for instance a light curve or spectrum per row. 
Those sub-tables can have different lengths for different objects, but Numpy and PyTorch
require rectangular arrays. 
LSDBstreamDataset provides a default mechanism for padding ragged nested columns,
but the default behavior may not suit all use cases.
The data batch will contain two keys for each nested column: one for the padded
array and one for the corresponding mask called `column_name` and `column_name_mask`.
If the default behavior does not suit your needs, you can implement a custom collator (see
[Dataset custom collation](../notebooks/custom_dataset_collation.ipynb)).

**Stopping an infinite stream.** Prefer breaking out of the session loop, as above. Calling
`stop()` from another thread also works, but LSDB offers no way to cancel a chunk that is
already being computed, so it takes effect only at the next chunk boundary. A smaller
`partitions_per_chunk` bounds that delay.